In [0]:
%sql
    
-- ============================================================================
-- USE UNITY CATALOG
-- ============================================================================
USE CATALOG adb_saas_dev;

-- ============================================================================
-- 1. CREATE LOGICAL SCHEMAS 
-- ============================================================================
CREATE SCHEMA IF NOT EXISTS silver
COMMENT 'Conformed and cleansed operational Delta Lake tables';

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Kimball Star Schema dimensional analytics layer';

CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Raw landing zone tables for MySQL, CSV, and API sources';

-- ============================================================================
-- 2. REGISTER MYSQL OLTP BRONZE TABLES
-- ============================================================================
CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_plans (
    plan_id INT,
    plan_name STRING,
    tier STRING,
    monthly_fee DECIMAL(10,2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_plans';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_organizations (
    org_id INT,
    name STRING,
    industry STRING,
    country STRING,
    is_active BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_organizations';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_users (
    user_id INT,
    org_id INT,
    email STRING,
    role STRING,
    is_active BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_users';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_subscriptions (
    subscription_id INT,
    org_id INT,
    plan_id INT,
    status STRING,
    started_at TIMESTAMP,
    cancelled_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_subscriptions';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_invoices (
    invoice_id INT,
    subscription_id INT,
    amount_due DECIMAL(10,2),
    amount_paid DECIMAL(10,2),
    status STRING,
    billing_date DATE,
    paid_at TIMESTAMP,
    updated_at TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_system STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/sql_invoices';

-- ============================================================================
-- 3. REGISTER FILE & API STREAM BRONZE TABLES
-- ============================================================================
CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_feature_usage (
    UserId STRING,
    OrgId STRING,
    UsageDate STRING,
    FeatureName STRING,
    UsageCount STRING,
    _ingested_at STRING,
    _source_file STRING,
    ingest_year INT,
    ingest_month INT
)
USING PARQUET
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/file_feature_usage';

CREATE TABLE IF NOT EXISTS adb_saas_dev.bronze.raw_logins (
    _ingested_at STRING,
    _source_file STRING,
    batch_id STRING,
    data ARRAY<STRUCT<client_device: STRING, event_id: STRING, event_timestamp: STRING, ip_address: STRING, login_success: BOOLEAN, user_uid: BIGINT>>,
    has_more BOOLEAN,
    page BIGINT,
    page_size BIGINT,
    record_count BIGINT,
    source STRING,
    status STRING,
    total_count BIGINT,
    ingest_year INT,
    ingest_month INT
)
USING JSON
LOCATION 'abfss://bronze@datalakesaas.dfs.core.windows.net/api_logins';